In [0]:
%run /Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# 1. dim_geography — one row per nation with parent region
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_geography AS
    SELECT
        n.n_nationkey   AS nation_key,
        n.n_name        AS nation_name,
        n.n_comment     AS nation_comment,
        r.r_regionkey   AS region_key,
        r.r_name        AS region_name,
        r.r_comment     AS region_comment
    FROM {catalog}.silver.nation n
    LEFT JOIN {catalog}.silver.region r
        ON n.n_regionkey = r.r_regionkey
""")
print(f"dim_geography: {spark.sql(f'SELECT COUNT(*) FROM {catalog}.gold.dim_geography').collect()[0][0]} rows")

# 2. dim_customer — one row per customer, denormalized with geography
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_customer AS
    SELECT
        c.c_custkey      AS customer_key,
        c.c_name         AS customer_name,
        c.c_address      AS address,
        c.c_phone        AS phone,
        c.c_acctbal      AS account_balance,
        c.c_mktsegment   AS market_segment,
        c.c_comment      AS comment,
        n.n_nationkey    AS nation_key,
        n.n_name         AS nation_name,
        r.r_regionkey    AS region_key,
        r.r_name         AS region_name
    FROM {catalog}.silver.customer c
    LEFT JOIN {catalog}.silver.nation n
        ON c.c_nationkey = n.n_nationkey
    LEFT JOIN {catalog}.silver.region r
        ON n.n_regionkey = r.r_regionkey
""")
print(f"dim_customer: {spark.sql(f'SELECT COUNT(*) FROM {catalog}.gold.dim_customer').collect()[0][0]:,} rows")

# 3. dim_product — one row per product
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_product AS
    SELECT
        p.p_partkey      AS product_key,
        p.p_name         AS product_name,
        p.p_mfgr         AS manufacturer,
        p.p_brand        AS brand,
        p.p_type         AS product_type,
        p.p_size         AS size,
        p.p_container    AS container,
        p.p_retailprice  AS retail_price,
        p.p_comment      AS comment
    FROM {catalog}.silver.part p
""")
print(f"dim_product: {spark.sql(f'SELECT COUNT(*) FROM {catalog}.gold.dim_product').collect()[0][0]:,} rows")

# 4. dim_supplier — one row per supplier, denormalized with geography
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_supplier AS
    SELECT
        s.s_suppkey      AS supplier_key,
        s.s_name         AS supplier_name,
        s.s_address      AS address,
        s.s_phone        AS phone,
        s.s_acctbal      AS account_balance,
        s.s_comment      AS comment,
        n.n_nationkey    AS nation_key,
        n.n_name         AS nation_name,
        r.r_regionkey    AS region_key,
        r.r_name         AS region_name
    FROM {catalog}.silver.supplier s
    LEFT JOIN {catalog}.silver.nation n
        ON s.s_nationkey = n.n_nationkey
    LEFT JOIN {catalog}.silver.region r
        ON n.n_regionkey = r.r_regionkey
""")
print(f"dim_supplier: {spark.sql(f'SELECT COUNT(*) FROM {catalog}.gold.dim_supplier').collect()[0][0]:,} rows")

In [0]:
# fact_sales — one row per order line with keys, measures, and attributes
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.fact_sales AS
    SELECT
        -- Keys (FKs to dimension tables)
        l.l_orderkey        AS order_key,
        l.l_linenumber      AS line_number,
        o.o_custkey         AS customer_key,
        l.l_partkey         AS product_key,
        l.l_suppkey         AS supplier_key,

        -- Measures
        l.l_quantity                                 AS quantity,
        l.l_extendedprice                           AS extended_price,
        l.l_discount                                AS discount,
        l.l_tax                                      AS tax,
        l.l_extendedprice * l.l_discount             AS discount_amount,
        l.l_extendedprice * (1 - l.l_discount)       AS net_price,
        l.l_extendedprice * (1 - l.l_discount) * l.l_tax      AS tax_amount,
        l.l_extendedprice * (1 - l.l_discount) * (1 + l.l_tax) AS net_revenue,
        o.o_totalprice                              AS order_total_price,

        -- Degenerate dimensions (order & line attributes)
        o.o_orderstatus    AS order_status,
        o.o_orderpriority  AS order_priority,
        o.o_orderdate      AS order_date,
        l.l_linestatus     AS line_status,
        l.l_returnflag     AS return_flag,
        l.l_shipdate       AS ship_date,
        l.l_commitdate     AS commit_date,
        l.l_receiptdate    AS receipt_date,
        l.l_shipmode       AS ship_mode,
        l.l_shipinstruct   AS ship_instruct

    FROM {catalog}.silver.lineitem l
    INNER JOIN {catalog}.silver.orders o
        ON l.l_orderkey = o.o_orderkey
""")
print(f"fact_sales: {spark.sql(f'SELECT COUNT(*) FROM {catalog}.gold.fact_sales').collect()[0][0]:,} rows")

In [0]:
# vw_annual_sales_kpis — annual revenue, orders, active customers, avg order value
# Definitions:
#   1. net_revenue (before tax) = SUM(extended_price * (1 - discount))  [= net_price in fact_sales]
#   2. orders                   = COUNT(DISTINCT order_key)
#   3. active_customers         = COUNT(DISTINCT customer_key) with >= 1 order in the year
#   4. avg_order_value          = net_revenue / COUNT(DISTINCT order_key)

spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_sales_kpis AS
    SELECT
        YEAR(f.order_date)                            AS sales_year,
        SUM(f.net_price)                              AS net_revenue,
        COUNT(DISTINCT f.order_key)                   AS total_orders,
        COUNT(DISTINCT f.customer_key)                AS active_customers,
        SUM(f.net_price) / COUNT(DISTINCT f.order_key) AS avg_order_value
    FROM {catalog}.gold.fact_sales f
    GROUP BY YEAR(f.order_date)
""")
print("vw_annual_sales_kpis created.\n")
display(spark.sql(f"SELECT * FROM {catalog}.gold.vw_annual_sales_kpis ORDER BY sales_year"))